## 1. Import Packages

In [3]:
from pathlib import Path
import platform
import sys
import time

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

In [ ]:
# allows the notebook to run from either the project root or notebooks/.
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# path to eLCS package
ELCS_ROOT = PROJECT_ROOT / "external" / "scikit-eLCS-master"

# checks if the eLCS package exists
if not ELCS_ROOT.exists():
    raise FileNotFoundError(
        f"Expected the supplied eLCS package at: {ELCS_ROOT}"
    )

# adds eLCS to import path
if str(ELCS_ROOT) not in sys.path:
    sys.path.insert(0, str(ELCS_ROOT))

from skeLCS import eLCS

# confirms the paths and import are successful
print("Project root:", PROJECT_ROOT)
print("eLCS source:", ELCS_ROOT)
print("eLCS imported successfully")

Project root: /Users/fatehbhular/Multi_Disease_Prediction_ENGE707
eLCS source: /Users/fatehbhular/Multi_Disease_Prediction_ENGE707/external/scikit-eLCS-master
eLCS imported successfully


In [ ]:
# checks the versions of python, NumPy, pandas, and scikit-learn
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.7
NumPy: 2.5.2
pandas: 3.0.5
scikit-learn: 1.9.0


## 2. Load the raw dataset

The baseline begins with the original `patient_data.csv` dataset. No feature engineering, scaling, outlier treatment, dimensionality reduction, or class-imbalance treatment is performed for the original baseline.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "patient_data.csv"

# loads the data from the csv dataset
df_raw = pd.read_csv(DATA_PATH)

# checks the shape, rows, columns, missing values, and duplicates
print("Raw shape:", df_raw.shape)
print("Raw records:", len(df_raw))
print("Raw columns:", df_raw.shape[1])
print("Missing values:", df_raw.isna().sum().sum())
print("Duplicate records:", df_raw.duplicated().sum())

# displays the first 5 rows for sanity check
display(df_raw.head())

Raw shape: (280985, 39)
Raw records: 280985
Raw columns: 39
Missing values: 0
Duplicate records: 2284


,composite_key,age_level,gender,bmi_level,smoking,diabetes,age,age_normalized,bmi,hypertension,...,salt_intake,heart_rate,hdl,ldl,education_level,employment_status,source_dataset,disease_flags,sublabel,label
0,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,9,0.080460,19.20,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
1,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.55,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
2,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.89,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
3,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,23.12,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
4,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,4,0.022989,19.61,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal


## 3. Construct the target variable

The original dataset does not contain `heart_disease_target`. The binary target is reconstructed using the target definition established in Phase I. Records with the disease labels `HT`, `DI_HT`, `HT_HY`, or `DI_HT_HY` are assigned to the positive class.

In [11]:
# all labels that include heart disease
HEART_DISEASE_LABELS = [
    "HT",
    "DI_HT",
    "HT_HY",
    "DI_HT_HY",
]

# copies the raw data and creates the binary target
df_baseline = df_raw.copy()

df_baseline["heart_disease_target"] = (
    df_baseline["sublabel"]
    .isin(HEART_DISEASE_LABELS)
    .astype(int)
)

# displays the target counts and verify the sublabel mapping.
print(df_baseline["heart_disease_target"].value_counts().sort_index())

display(
    pd.crosstab(
        df_baseline["sublabel"],
        df_baseline["heart_disease_target"],
    )
)

heart_disease_target
0    275081
1      5904
Name: count, dtype: int64


heart_disease_target,0,1
sublabel,,
DI,31918,0
DI_HT,0,1398
DI_HT_HY,0,845
DI_HY,66473,0
HT,0,2595
HT_HY,0,1066
HY,69806,0
N,106884,0


## 5. Apply miminum baseline processing

This is not full preprocessing, it only prevents leakage and makes the data usable. There is no scalingm outlier handling, feature engineering, dimension reduction, or imbalance correction in this baseline.

In [ ]:
# number of records before removing duplicates
records_before = len(df_baseline)

# removes the duplicate rows and reindexes all rows
df_baseline = df_baseline.drop_duplicates(
    subset=df_raw.columns.tolist()
).reset_index(drop=True)

# shows the number of duplicates removed
print("Duplicates removed:", records_before - len(df_baseline))

Duplicates removed: 2284


In [13]:
# the target-related columns that cannot be predictors
EXCLUDED_COLUMNS = [
    "heart_disease_target",
    "sublabel",
    "disease_flags",
    "label",
    "heart_disease",
    "composite_key",
    "source_dataset",
]

# separates the predictor variables from the target
X = df_baseline.drop(columns=EXCLUDED_COLUMNS)
y = df_baseline["heart_disease_target"].astype(int)

# displays the feature and target dimensions
print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("Positive-class rate:", y.mean())

Predictor shape: (278701, 33)
Target shape: (278701,)
Positive-class rate: 0.021137347910484713


## 6. Identify categorical columns

We need a numeric NumPy feature array. Categorical columns must be converted to numbers.

In [15]:
# identifies the categorical columns that need to be converted to numerical
categorical_columns = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

# displays the categorical columns
print("Categorical columns:", categorical_columns)

Categorical columns: ['age_level', 'gender', 'bmi_level', 'smoking', 'diabetes', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'high_blood_pressure', 'sugar_consumption', 'education_level', 'employment_status']


## 7. Create an 80/20 split

We need either an 80/20 split or k-fold cross-validation.

In [16]:
# select one reproducible seed for the initial baseline check.
SEED = 42

# create a shuffled and stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
    shuffle=True,
)

# displays the partition sizes and positive-class proportions.
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

Training shape: (222960, 33)
Testing shape: (55741, 33)
Training positive rate: 0.02113832077502691
Testing positive rate: 0.021133456522129133


## 8. Converting Categories to numeric values

Fit the encoder only on the training data to prevent test-data leakage.

scikit-eLCS 1.2.4 cannot process text values directly. Therefore every categorical value must be converted into a number.

In [ ]:
# creates an encoder that can handle unseen test-set categories
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

# copies the test/train sets before replacing categorical columns
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# learns category mappings from the training data and applies them to both partitions.
X_train_encoded[categorical_columns] = encoder.fit_transform(
    X_train[categorical_columns]
)
X_test_encoded[categorical_columns] = encoder.transform(
    X_test[categorical_columns]
)

# converts pandas objects into NumPy arrays (required by eLCS)
dataFeatures_train = X_train_encoded.to_numpy(dtype=float)
dataFeatures_test = X_test_encoded.to_numpy(dtype=float)
dataPhenotypes_train = y_train.to_numpy(dtype=int)
dataPhenotypes_test = y_test.to_numpy(dtype=int)

# keeps the feature names for later rule-population
dataHeaders = X.columns.to_numpy()

# confirms the arrays meet input requirements
print("Training features:", dataFeatures_train.shape)
print("Training target:", dataPhenotypes_train.shape)
print("Testing features:", dataFeatures_test.shape)
print("Testing target:", dataPhenotypes_test.shape)

assert np.isfinite(dataFeatures_train).all()
assert np.isfinite(dataFeatures_test).all()

Training features: (222960, 33)
Training target: (222960,)
Testing features: (55741, 33)
Testing target: (55741,)


## 9. Initialise the original eLCS baseline

Parameters are kept the same for the original baseline.

In [ ]:
# creates the eLCS object using the constructor
model = eLCS()

In [ ]:
# creates the original eLCS model using version 1.2.4 defaults.
model = eLCS(random_state=SEED)

# displays every parameter actually used by the baseline.
parameter_table = pd.DataFrame(
    [
        {"Parameter": name, "Value": value}
        for name, value in model.get_params().items()
    ]
)

display(parameter_table)

,Parameter,Value
0,N,1000
1,acc_sub,0.99
2,beta,0.2
3,chi,0.8
4,delta,0.1
5,discrete_attribute_limit,10
6,do_GA_subsumption,True
7,do_correct_set_subsumption,False
8,fitness_reduction,0.1
9,init_fit,0.01


## 10. Model Training

Record the start time and train the oringnal eLCS model. Then display the total baseline training time. 

In [ ]:
# records the start time and trains the original eLCS model.
start_time = time.perf_counter()

trainedModel = model.fit(
    dataFeatures_train,
    dataPhenotypes_train,
)

# training time in seconds
training_seconds = time.perf_counter() - start_time

# displays the total baseline training time and the number of iterations
print("Training time in seconds:", training_seconds)
print(trainedModel.learning_iterations)

Training time in seconds: 8.816207709023729
10000


## 11. Generate predictions

Using `predict()` and `predict_proba()` to make predictions and calculate probabilities.

In [22]:
# predicts the binary class for every testing set observation.
predictions = trainedModel.predict(dataFeatures_test)

# calculates eLCS voting probabilities for both target classes.
probabilities = trainedModel.predict_proba(dataFeatures_test)

# selects the probabilities corresponding to positive class 1.
positive_probabilities = probabilities[:, 1]

# converts the predictions into an integer NumPy array.
predictions = np.asarray(predictions).astype(int)

# displays the example predictions and probabilities.
print("First predictions:", predictions[:10])
print("First probabilities:", probabilities[:10])

First predictions: [0 0 0 0 0 0 0 0 0 0]
First probabilities: [[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]


## 12. Calculate basline metrics

Approximately 2.1% of records are positive for heart disease, so accuracy alone is insufficient.

In [ ]:
# calculates the classification metrics for the imbalanced binary target.
# the target is imbalanced because there are way more "no heart disease" cases than "heart disease" cases.
accuracy = accuracy_score(dataPhenotypes_test, predictions)

# precision measurers how often a heart disease prediction is correct.
precision = precision_score(
    dataPhenotypes_test,
    predictions,
    zero_division=0,
)

# recall measurers how often a heart disease prediction is correct
recall = recall_score(
    dataPhenotypes_test,
    predictions,
    zero_division=0,
)

# F1 combines precision and recall into one score
f1 = f1_score(
    dataPhenotypes_test,
    predictions,
    zero_division=0,
)

# balanced accuracy measures accuracy for both classes separately
balanced_accuracy = balanced_accuracy_score(
    dataPhenotypes_test,
    predictions,
)

# ROC-AUC measures how well the model can separate the two classes
roc_auc = roc_auc_score(
    dataPhenotypes_test,
    positive_probabilities,
)

# PR-AUC measures the balance between precision and recall
pr_auc = average_precision_score(
    dataPhenotypes_test,
    positive_probabilities,
)

# calculates the confusion matrix using class order 0, then 1.
matrix = confusion_matrix(
    dataPhenotypes_test,
    predictions,
    labels=[0, 1],
)

# places the metrics into a readable results table
baseline_results = pd.DataFrame(
    {
        "Metric": [
            "Accuracy", "Precision", "Recall", "F1", "Balanced accuracy", "ROC-AUC", "PR-AUC", "Training seconds",
        ],
        "Value": [
            accuracy, precision, recall, f1, balanced_accuracy, roc_auc, pr_auc, training_seconds,
        ],
    }
)

# display the classification metrics
display(baseline_results)

# display the confusion matrix with labels showing
display(
    pd.DataFrame(
        matrix,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"],
    )
)

,Metric,Value
0,Accuracy,0.978867
1,Precision,0.000000
2,Recall,0.000000
3,F1,0.000000
4,Balanced accuracy,0.500000
5,ROC-AUC,0.589726
6,PR-AUC,0.042961
7,Training seconds,8.816208


,Predicted 0,Predicted 1
Actual 0,54563,0
Actual 1,1178,0


## 13. Inspect eLCS-specific basline information

In [ ]:
# collects the eLCS-specific diagnostics
print(
    "Final training balanced accuracy:",
    trainedModel.get_final_accuracy(),
)
print(
    "Final training coverage:",
    trainedModel.get_final_instance_coverage(),
)
print(
    "Final attribute specificity:",
    trainedModel.get_final_attribute_specificity_list(),
)
print(
    "Final attribute accuracy:",
    trainedModel.get_final_attribute_accuracy_list(),
)

Final training balanced accuracy: 0.5
Final training coverage: 1.0
Final attribute specificity: [287, 283, 274, 276, 333, 333, 351, 359, 395, 356, 328, 366, 295, 334, 321, 314, 324, 331, 323, 386, 294, 287, 489, 434, 366, 310, 310, 337, 338, 319, 362, 268, 316]
Final attribute accuracy: [281.7691308153629, 268.209006587017, 260.1241068281584, 259.9253160948819, 315.6903898867182, 321.5135198907323, 334.2818508106286, 349.0192288482311, 386.5891470148557, 345.72215510888395, 316.7237776804575, 349.26286047227836, 285.1403659595559, 321.7285267383889, 304.07840178528767, 297.1749753278973, 311.56134322902625, 320.58219919176355, 307.3812708139978, 376.7434897952799, 283.0380682714977, 276.90760914228815, 469.4074585741699, 417.60217832282353, 354.21560923734336, 296.3978474515662, 301.5492710862868, 321.3355284730461, 319.4475778031161, 304.68691427324524, 349.0199080582189, 260.96889262246583, 301.93150588462765]


## 14. Repeating the basline 10 times

In [29]:
# 10 reproducible seeds for the repeated experiements.
SEEDS = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]

In [30]:
# runs the complete original eLCS baseline using a seed
def run_baseline(seed):

    # creates a stratified 80/20 train-test split.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y,
        shuffle=True,
    )

    # creates the training-based categorical encoder.
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    # copies the partitions before encoding categorical columns.
    X_train_encoded = X_train.copy()
    X_test_encoded = X_test.copy()

    # learns mappings from training data and apply them to both partitions.
    X_train_encoded[categorical_columns] = encoder.fit_transform(
        X_train[categorical_columns]
    )
    X_test_encoded[categorical_columns] = encoder.transform(
        X_test[categorical_columns]
    )

    # converts the features and targets into numeric NumPy arrays.
    train_features = X_train_encoded.to_numpy(dtype=float)
    test_features = X_test_encoded.to_numpy(dtype=float)
    train_target = y_train.to_numpy(dtype=int)
    test_target = y_test.to_numpy(dtype=int)

    # initialises the original eLCS defaults using the current seed.
    model = eLCS(random_state=seed)

    # trains the model and measure its training time.
    start_time = time.perf_counter()
    model.fit(train_features, train_target)
    training_seconds = time.perf_counter() - start_time

    # generates class predictions and positive-class voting scores.
    predictions = np.asarray(
        model.predict(test_features)
    ).astype(int)

    probabilities = model.predict_proba(
        test_features
    )[:, 1]

    # extracts the four confusion-matrix values.
    tn, fp, fn, tp = confusion_matrix(
        test_target,
        predictions,
        labels=[0, 1],
    ).ravel()

    # returns the metrics and confusion-matrix results.
    return {
        "Seed": seed,
        "Accuracy": accuracy_score(
            test_target,
            predictions,
        ),
        "Precision": precision_score(
            test_target,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            test_target,
            predictions,
            zero_division=0,
        ),
        "F1": f1_score(
            test_target,
            predictions,
            zero_division=0,
        ),
        "Balanced accuracy": balanced_accuracy_score(
            test_target,
            predictions,
        ),
        "ROC-AUC": roc_auc_score(
            test_target,
            probabilities,
        ),
        "PR-AUC": average_precision_score(
            test_target,
            probabilities,
        ),
        "True negatives": tn,
        "False positives": fp,
        "False negatives": fn,
        "True positives": tp,
        "Positive predictions": int(
            (predictions == 1).sum()
        ),
        "Training seconds": training_seconds,
    }

In [ ]:
# stores the result from each seed.
baseline_runs = []

# run the unchanged basline once for every seed.
for seed in SEEDS:
    print(f"Starting seed {seed}..")

    result = run_baseline(seed)
    baseline_runs.append(result)

    # displays performance and the number of positive predictions after each run.
    print(
        f"Completed seed {seed}: "
        f"balanced accuracy={result['Balanced accuracy']:.4f}, "
        f"ROC-AUC={result['ROC-AUC']:.4f}, "
        f"PR-AUC={result['PR-AUC']:.4f}, "
        f"positive predictions={result['Positive predictions']}"
    )

# combines the 10 results into one table
basline_runs_df = pd.DataFrame(baseline_runs)

display(basline_runs_df)

Starting seed 3..
Completed seed 3: balanced accuracy=0.5000, ROC-AUC=0.5653, PR-AUC=0.0405, positive predictions=0
Starting seed 6..
Completed seed 6: balanced accuracy=0.5000, ROC-AUC=0.6289, PR-AUC=0.0636, positive predictions=0
Starting seed 9..
Completed seed 9: balanced accuracy=0.5000, ROC-AUC=0.5703, PR-AUC=0.0357, positive predictions=0
Starting seed 12..
Completed seed 12: balanced accuracy=0.5000, ROC-AUC=0.5602, PR-AUC=0.0369, positive predictions=0
Starting seed 15..
Completed seed 15: balanced accuracy=0.5000, ROC-AUC=0.5844, PR-AUC=0.0432, positive predictions=0
Starting seed 18..
Completed seed 18: balanced accuracy=0.5000, ROC-AUC=0.5624, PR-AUC=0.0440, positive predictions=0
Starting seed 21..
Completed seed 21: balanced accuracy=0.5000, ROC-AUC=0.5799, PR-AUC=0.0348, positive predictions=0
Starting seed 24..
Completed seed 24: balanced accuracy=0.5000, ROC-AUC=0.5151, PR-AUC=0.0298, positive predictions=0
Starting seed 27..
Completed seed 27: balanced accuracy=0.5000

,Seed,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC,True negatives,False positives,False negatives,True positives,Positive predictions,Training seconds
0,3,0.978867,0.0,0.0,0.0,0.5,0.565332,0.040503,54563,0,1178,0,0,12.868947
1,6,0.978867,0.0,0.0,0.0,0.5,0.628920,0.063617,54563,0,1178,0,0,12.603986
2,9,0.978867,0.0,0.0,0.0,0.5,0.570289,0.035747,54563,0,1178,0,0,12.457920
3,12,0.978867,0.0,0.0,0.0,0.5,0.560207,0.036945,54563,0,1178,0,0,12.846011
4,15,0.978867,0.0,0.0,0.0,0.5,0.584409,0.043152,54563,0,1178,0,0,13.037642
5,18,0.978867,0.0,0.0,0.0,0.5,0.562358,0.044012,54563,0,1178,0,0,12.878013
6,21,0.978867,0.0,0.0,0.0,0.5,0.579893,0.034822,54563,0,1178,0,0,12.824967
7,24,0.978867,0.0,0.0,0.0,0.5,0.515064,0.029793,54563,0,1178,0,0,12.662041
8,27,0.978867,0.0,0.0,0.0,0.5,0.605667,0.047448,54563,0,1178,0,0,12.787320
9,30,0.978867,0.0,0.0,0.0,0.5,0.580275,0.051921,54563,0,1178,0,0,12.807647
